# VAN-FED-IDS trên CICIoV — class-incremental

Chen et al., *Fast and practical intrusion detection system based on federated
learning for VANET*, Computers & Security 142 (2024) 103881.

Chạy nối tiếp task 0→4, resume giữa các task, round đánh số liên tục.

**Cần trước khi chạy:** một Kaggle Dataset tên `iov-100client` chứa nguyên thư mục
`100client` (giữ thư mục con `federated_data/`). Bật **GPU T4** trong Settings.

## 1. Setup

In [ ]:
import os, glob, time

REPO = "VANFED-IDS"
CODE = f"/kaggle/working/{REPO}"
DATA = "/kaggle/input/iov-100client"     # sửa cho khớp tên Dataset của bạn

if not os.path.isdir(CODE):
    !git clone -q https://github.com/TongXuanVu/{REPO}.git {CODE}
else:
    !cd {CODE} && git pull -q

!pip install -q flwr
import flwr, torch
print("flwr", flwr.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.is_available())

## 2. Kiểm tra dữ liệu — đừng bỏ qua

In [ ]:
fed = os.path.join(DATA, "federated_data")
assert os.path.isdir(fed), f"Khong thay {fed}. Sua bien DATA, hoac Dataset chua gan vao notebook."

shards = sorted(glob.glob(os.path.join(fed, "*.pt")))
print(f"{len(shards)} shard (ky vong 500 cho bo 100client)")

blob = torch.load(os.path.join(DATA, "global_test_data.pt"),
                  map_location="cpu", weights_only=False)
x, y = blob["x"], blob["y"]
print(f"global test : x={tuple(x.shape)} {x.dtype} | so lop={int(y.max()) + 1}")

assert x.shape[1] == 31, f"So dac trung = {x.shape[1]}, khong phai 31 -> phai sua INPUT_LEN"
assert int(y.max()) + 1 <= 13, "Nhieu hon 13 lop -> phai sua NUM_GLOBAL_CLASSES"
print("\nDu lieu OK.")

## 3. Chạy thử nhanh (~3 phút)

Bắt buộc. Nếu dữ liệu thật khác định dạng, cell này lộ ra ngay thay vì để bạn
mất hai tiếng rồi mới báo lỗi.

In [ ]:
!cd {CODE} && python run_fl.py --data-dir {DATA} \
    --out-dir /kaggle/working/_thu --clients 3 --rounds 2 --tasks 0,1 \
    --max-samples 20000 --test-samples 20000 --server-warmup 20 2>&1 | tail -20

## 4. Chạy thật — **cứ chạy lại cell này mỗi khi Kaggle hết giờ**

Mỗi round đều ghi thẳng xuống đĩa ngay: `metrics_task*.csv`, `checkpoints/round_NNN.pth`,
`_logs/*.log`. Session bị cắt giữa chừng không mất gì.

Session sau, **chạy lại đúng cell này**: nó đếm số dòng trong CSV, bỏ qua task đã đủ
round, và chỉ chạy tiếp số round còn thiếu của task đang dở. Round vẫn đánh liên tục,
CSV không có dòng trùng.

`--cm-every 5` ghi confusion matrix mỗi 5 round, để bị cắt giữa task vẫn có bản gần nhất.

> Muốn xoá sạch làm lại: đổi `--out-dir`, hoặc xoá thư mục đó rồi thêm `--restart`.


In [ ]:
t0 = time.time()
!cd {CODE} && python run_fl.py --data-dir {DATA} \
    --out-dir /kaggle/working/out_p1 \
    --clients 10 --rounds 30 --batch-size 512 --server-warmup 25 --cm-every 5
print(f"\nXong sau {(time.time() - t0) / 60:.1f} phut")

In [ ]:
# Đối chứng: FL thường (gộp cả 5 task, không class-incremental)
t0 = time.time()
!cd {CODE} && python run_fl.py --data-dir {DATA} \
    --out-dir /kaggle/working/out_p1_flat --tasks none \
    --clients 10 --rounds 30 --batch-size 512 --server-warmup 25 --cm-every 5
print(f"\nXong sau {(time.time() - t0) / 60:.1f} phut")

## 5. Gộp kết quả + đo mức độ quên

In [ ]:
!cd {CODE} && python collect_results.py --out-dir /kaggle/working/ket_qua \
    --runs P1-incremental=/kaggle/working/out_p1 \
           P1-flat=/kaggle/working/out_p1_flat

In [ ]:
import pandas as pd
from IPython.display import display, Image

K = "/kaggle/working/ket_qua"
print("=== So sanh ===");     display(pd.read_csv(f"{K}/comparison.csv"))
print("=== Muc do quen ==="); display(pd.read_csv(f"{K}/forgetting.csv"))
display(Image(f"{K}/accuracy_curve.png"))
display(Image(f"{K}/forgetting_P1-incremental.png"))

## 6. Đóng gói tải về

In [ ]:
# Kiem tra truoc khi tai ve: du dong metric? du checkpoint? du log?
import glob, os

for f in sorted(glob.glob("/kaggle/working/ket_qua/metrics_all_*.csv")):
    n = sum(1 for _ in open(f)) - 1
    print(f"{os.path.basename(f):32s} {n:4d} dong metric  (ky vong 150 = 30 round x 5 task)")

for d in sorted(glob.glob("/kaggle/working/out_*/checkpoints*")):
    print(f"{d:52s} {len(glob.glob(d + '/round_*.pth')):4d} checkpoint")

print(f"\n{len(glob.glob('/kaggle/working/out_*/_logs/*.log'))} file log")
print(f"{len(glob.glob('/kaggle/working/out_*/confusion_matrix_*.csv'))} confusion matrix CSV")


In [ ]:
# Dong goi DAY DU: checkpoint (.pth) + log + CSV + confusion matrix
!cd /kaggle/working && zip -qr ket_qua_p1.zip ket_qua out_p1 out_p1_flat
!du -sh /kaggle/working/out_p1 /kaggle/working/out_p1_flat
!ls -lh /kaggle/working/ket_qua_p1.zip
print("\nTai ve tu tab Output ben phai TRUOC KHI het session.")
